# Affine air-gap ARC-AGI-2 submit

Internet **disabled**. No solving online.

- Embeds verified local eval mastery (**120 tasks / 172 grids**, FoT SHA pinned in MANIFEST).
- Emits platform `submission.json` for official **test** challenges (**240 tasks**).
- Pattern matches top-score notebooks: write `/kaggle/working/submission.json` only.

**Steward:** Run All → Save Version → Submit to competition (after UTC daily quota reset).
Do **not** use direct `kaggle competitions submit` for this track (Notebooks only).


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
from pathlib import Path

WORKING = Path("/kaggle/working")
SRC_CANDIDATES = [
    Path("/kaggle/working/payload/submission.json"),  # after local staging
    Path("payload/submission.json"),
    Path("/kaggle/input") / "affine-agi2-airgap-payload" / "submission.json",
]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()


def load_platform_submission() -> dict:
    for cand in SRC_CANDIDATES:
        if cand.is_file():
            print(f"loading platform payload: {cand}")
            return json.loads(cand.read_text(encoding="utf-8"))
    b64 = Path("embedded_submission_b64.txt")
    if b64.is_file():
        print(f"loading platform payload from {b64}")
        return json.loads(base64.b64decode(b64.read_text(encoding="utf-8")))
    raise FileNotFoundError(
        "No embedded AGI-2 platform submission.json found in kernel files."
    )


submission = load_platform_submission()
assert isinstance(submission, dict) and submission, "empty submission"
for task_id, attempts in submission.items():
    assert isinstance(attempts, list) and attempts, task_id
    for item in attempts:
        assert set(item) == {"attempt_1", "attempt_2"}, (task_id, set(item))

out = WORKING / "submission.json"
out.write_text(json.dumps(submission, separators=(",", ":")), encoding="utf-8")
print(
    f"Wrote {out} tasks={len(submission)} grids={sum(len(v) for v in submission.values())} "
    f"sha256={sha256(out)}"
)

fot = Path("payload/fot_eval_mastery_submission.json")
if fot.is_file():
    print(
        f"FoT eval mastery sidecar present: tasks={len(json.loads(fot.read_text()))} "
        f"sha256={sha256(fot)} (not the competition filename)"
    )

# Optional: confirm competition test IDs if mounted
input_root = Path("/kaggle/input")
if input_root.is_dir():
    hits = list(input_root.rglob("arc-agi_test_challenges.json"))
    if hits:
        challenges = json.loads(hits[0].read_text(encoding="utf-8"))
        missing = sorted(set(challenges) - set(submission))
        extra = sorted(set(submission) - set(challenges))
        print(f"test_challenges={len(challenges)} missing={len(missing)} extra={len(extra)}")
        if missing or extra:
            raise RuntimeError(f"submission keys mismatch competition test set: missing={missing[:5]} extra={extra[:5]}")
        print("platform key coverage OK vs arc-agi_test_challenges.json")
